### **13. Model Inference - Klasifikasi Otomatis Kategori Berita Indonesia Menggunakan Artificial Neural Network (ANN)**

Notebook ini digunakan untuk menguji model ensemble terbaik (gabungan dari 4 model ANN) yang telah dilatih pada notebook utama. Pengujian dilakukan menggunakan data teks berita baru yang belum pernah dilihat oleh model sebelumnya.
  * Data yang digunakan pada notebook ini merepresentasikan skenario nyata, di mana sistem menerima judul teks mentah. Proses inferensi akan mereplikasi tahapan preprocessing (pembersihan teks, tokenisasi, dan padding) persis seperti yang dilakukan pada saat training agar model dapat menghasilkan prediksi kategori yang akurat.

#### **13.1 Import Library**
Pada tahap pertama, memanggil semua library yang dibutuhkan untuk memproses teks, melakukan tokenisasi, serta memuat model Deep Learning berbasis Keras/TensorFlow.

In [6]:
# ============================================================
# Import Library yang Dibutuhkan untuk Model Inference
# ============================================================

import re
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print("✅ Library berhasil diimport")

✅ Library berhasil diimport


#### **13.2 Load Model dan Artifacts**
Model final menggunakan pendekatan ensemble dari empat arsitektur ANN yang berbeda untuk mengambil rata-rata probabilitasnya. Selain model, diperlukan juga memuat tokenizer dan label_encoder yang sudah di-fit pada data training.

In [2]:
# ============================================================
# Load Model Keras dan Artifacts (Tokenizer & Label Encoder)
# ============================================================

# 1. Load Model ANN
model_2 = load_model('saved_model/model_2_baseline.keras')
model_2b = load_model('saved_model/model_2b_numfilters128.keras')
model_2c = load_model('saved_model/model_2c_dropout03.keras')
model_3 = load_model('saved_model/model_3_word2vec.keras')

ensemble_models = [model_2, model_2b, model_2c, model_3]

# 2. Load Tokenizer & Label Encoder
with open('saved_model/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

with open('saved_model/label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
    
# Tetapkan MAX_LENGTH sesuai dengan yang digunakan saat training
MAX_LENGTH = 15

print("✅ Seluruh model ensemble dan artifact berhasil dimuat")
print(f"Kelas target: {list(label_encoder.classes_)}")

✅ Seluruh model ensemble dan artifact berhasil dimuat
Kelas target: ['finance', 'health', 'oto', 'sport', 'travel']


#### **13.3 Data Berita Baru**
Berikut adalah contoh data judul berita baru yang mewakili berbagai topik seperti keuangan, kesehatan, otomotif, olahraga, dan traveling. Data ini akan dimasukkan ke dalam bentuk DataFrame untuk mempermudah visualisasi hasil.

In [3]:
# ============================================================
# Data Teks Baru (Format Mentah/Raw)
# ============================================================

contoh_judul = [
    "harga emas hari ini naik tipis di tengah ketidakpastian ekonomi global",
    "menteri kesehatan minta masyarakat waspada gejala flu singapura",
    "test drive mobil listrik terbaru dengan jarak tempuh 500 km",
    "timnas indonesia menang telak di laga kualifikasi piala dunia",
    "tips memilih destinasi wisata keluarga saat libur panjang"
]

data_baru = pd.DataFrame({'Judul_Berita': contoh_judul})

print("Data berita baru (raw):")
display(data_baru)

Data berita baru (raw):


,Judul_Berita
0,harga emas hari ini naik tipis di tengah ketid...
1,menteri kesehatan minta masyarakat waspada gej...
2,test drive mobil listrik terbaru dengan jarak ...
3,timnas indonesia menang telak di laga kualifik...
4,tips memilih destinasi wisata keluarga saat li...


#### **13.4 Replikasi Preprocessing**
Fungsi clean_text harus diimplementasikan dengan logika yang sama persis dengan yang digunakan di notebook utama (misalnya menghapus tanda baca, angka, atau melakukan lowercasing).

In [4]:
# ============================================================
# Replikasi Fungsi Cleaning Text
# Sesuaikan regex di bawah dengan proses cleaning di notebook utama
# ============================================================
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = stopword_remover.remove(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Terapkan cleaning ke data baru
data_baru['Judul_Clean'] = data_baru['Judul_Berita'].apply(clean_text)

print("Data setelah melalui proses Text Cleaning:")
display(data_baru[['Judul_Berita', 'Judul_Clean']])

Data setelah melalui proses Text Cleaning:


,Judul_Berita,Judul_Clean
0,harga emas hari ini naik tipis di tengah ketid...,harga emas hari naik tipis tengah ketidakpasti...
1,menteri kesehatan minta masyarakat waspada gej...,menteri kesehatan minta masyarakat waspada gej...
2,test drive mobil listrik terbaru dengan jarak ...,test drive mobil listrik terbaru jarak tempuh km
3,timnas indonesia menang telak di laga kualifik...,timnas indonesia menang telak laga kualifikasi...
4,tips memilih destinasi wisata keluarga saat li...,tips memilih destinasi wisata keluarga libur p...


#### **13.5 Prediksi Kategori Berita**
Fungsi prediksi akan mengubah teks yang sudah bersih menjadi sekuens angka, memotong/menambahkan padding, lalu memasukkannya ke dalam keempat model. Nilai probabilitas akan dirata-rata untuk menentukan kategori final.

In [5]:
# ============================================================
# Prediksi Kategori Menggunakan Model Ensemble
# ============================================================

def predict_category_batch(texts, models, tokenizer, label_encoder, max_length):
    texts_clean = [clean_text(t) for t in texts]

    seq = tokenizer.texts_to_sequences(texts_clean)
    pad = pad_sequences(seq, maxlen=max_length, padding="post", truncating="post")

    proba_list = [model.predict(pad, verbose=0) for model in models]
    proba_avg = np.mean(proba_list, axis=0)

    pred_idx = np.argmax(proba_avg, axis=1)
    pred_labels = label_encoder.inverse_transform(pred_idx)
    confidences = np.max(proba_avg, axis=1)

    return texts_clean, pred_labels, confidences, proba_avg

# Eksekusi fungsi prediksi pada kolom judul yang sudah dibersihkan
judul_clean_hasil, labels, confs, proba_matrix = predict_category_batch(
    data_baru['Judul_Berita'].tolist(),
    ensemble_models,
    tokenizer,
    label_encoder,
    MAX_LENGTH
)

# Simpan hasil prediksi ke dalam DataFrame
hasil_inference = pd.DataFrame({
    'Judul_Berita': data_baru['Judul_Berita'],
    'Prediksi_Kategori': labels,
    'Confidence': confs
})

# Tambahkan matriks probabilitas untuk setiap kelas (opsional, untuk detail)
df_proba = pd.DataFrame(proba_matrix, columns=label_encoder.classes_)
hasil_inference = pd.concat([hasil_inference, df_proba], axis=1)

# Tandai prediksi dengan confidence di bawah 60% untuk tinjauan manual (opsional)
hasil_inference['Perlu_Review_Manual'] = hasil_inference['Confidence'] < 0.60

print("=" * 85)
print("\t Hasil Prediksi Kategori Berita Otomatis (Ensemble ANN)")
print("=" * 85)
display(hasil_inference)

	 Hasil Prediksi Kategori Berita Otomatis (Ensemble ANN)


,Judul_Berita,Prediksi_Kategori,Confidence,finance,health,oto,sport,travel,Perlu_Review_Manual
0,harga emas hari ini naik tipis di tengah ketid...,finance,0.994644,0.994644,0.000344,4.517486e-04,4.969626e-05,0.004511,False
1,menteri kesehatan minta masyarakat waspada gej...,health,0.994534,0.004037,0.994534,1.778782e-04,3.144714e-04,0.000936,False
2,test drive mobil listrik terbaru dengan jarak ...,oto,0.999466,0.000395,0.000028,9.994656e-01,4.115732e-05,0.000070,False
3,timnas indonesia menang telak di laga kualifik...,sport,0.999656,0.000002,0.000009,2.670284e-04,9.996557e-01,0.000066,False
4,tips memilih destinasi wisata keluarga saat li...,travel,0.999978,0.000016,0.000004,1.348062e-07,5.120874e-07,0.999978,False
